# Part 2 — MYH Curated Applications Dataset (2020–2025)

**Notebook role:** This notebook will become the full, rerunnable raw-to-curated workflow for Part 2 of the Data Pipeline Project.

**Current implementation status:**  
Sub-project **2.5** has now implemented the cleaning, normalization, and thoughtful enrichment layer on top of the reusable 2.4 ingestion base. That 2.4 base already keeps `seqf_niva`, `sokta_platser_per_utbildningsomgang`, `sokta_platser_totalt`, and `beviljade_platser_totalt` concat-stable as nullable `Int64` before cross-year combination. On that inherited stable base, the notebook creates `cleaned_applications` through a copy-based cleaning function, applies conservative whitespace cleanup to configured text fields, revalidates the sparse integer dtype contract, then creates `curated_applications` with populated `beslut_normalized`, `huvudmannatyp_normalized`, `is_approved`, `is_distance_based`, and `has_multiple_municipalities`. The curated working table remains **7,641 rows × 32 columns** and now includes transformation-scoped integrity checks for rerun safety, input-table preservation, dtype intent, schema order, row-count preservation, and allowed normalization domains.  
Broader dataset validation, missingness review, final export, and final SQL/API-facing polish remain intentionally reserved for Sub-projects **2.6–2.7**.

## Project purpose

The finished notebook should:
- read the original MYH Excel workbooks for application rounds **2020–2025**,
- build a longitudinal curated applications dataset from **`Tabell 3`**,
- preserve a clear main-table grain: **one row = one application in one application round**,
- document source differences, harmonization choices, cleaning choices, enrichment, and validation,
- export a final dataset that can later be loaded into SQL and served through a read-oriented API.


## 1. Scope and design principles

### Fixed project direction
- Source years: **2020, 2021, 2022, 2023, 2024, 2025**
- Main source sheet: **`Tabell 3`**
- Main table grain: **one application in one application round per row**
- `Tabell 4` is acknowledged as useful but must **not** be merged into the main applications table in a way that duplicates applications.

### Working quality standard
This notebook should read as an explanatory data journey rather than a code dump.  
Every major transformation will later be accompanied by:
1. what is being done,
2. why it is being done,
3. what trade-off or source inconsistency it addresses.


## 2. Imports and runtime setup

This cell contains the small set of general-purpose libraries expected in the notebook.  
Additional imports should only be added later when they support a concrete implementation need.


In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 100)

## 3. Project paths and folder conventions

The notebook is designed to work whether VS Code runs it from:
- the repository root, or
- the `part_2/` folder itself.

The raw-vs-processed rule is simple:
- `data/raw/` contains the unchanged MYH Excel inputs,
- `data/processed/` contains notebook-generated exports.


In [ ]:
def resolve_part_2_dir() -> Path:
    """Return the Part 2 workspace when run from repo root or from part_2/."""
    cwd = Path.cwd().resolve()

    if cwd.name == "part_2":
        return cwd

    candidate = cwd / "part_2"
    if candidate.exists():
        return candidate

    raise FileNotFoundError(
        "Could not locate the 'part_2' folder. "
        "Run this notebook from the repository root or from the part_2/ folder."
    )


PART_2_DIR = resolve_part_2_dir()
RAW_DATA_DIR = PART_2_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PART_2_DIR / "data" / "processed"

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"Part 2 workspace: {PART_2_DIR}")
print(f"Raw input folder: {RAW_DATA_DIR}")
print(f"Processed export folder: {PROCESSED_DATA_DIR}")


## 4. Raw-data inventory check

This small setup check confirms which Excel files are currently present in `data/raw/`.  
The full workbook and sheet exploration belongs to **Sub-project 2.2**, but this inventory makes the workspace immediately usable.


In [3]:
EXPECTED_SOURCE_YEARS = [2020, 2021, 2022, 2023, 2024, 2025]
SOURCE_YEAR_RE = re.compile(r"(20\d{2})")

raw_excel_files = sorted(RAW_DATA_DIR.glob("*.xlsx"))

print(f"Excel workbooks currently found: {len(raw_excel_files)}")
for file_path in raw_excel_files:
    print(f"- {file_path.name}")

if not raw_excel_files:
    print(
        "\nNo raw Excel files have been found yet. "
        "Place the six original MYH 2020–2025 workbooks in data/raw/ "
        "before starting the source exploration section."
    )

Excel workbooks currently found: 6
- resultat-ansokningsomgang-2020.xlsx
- resultat-ansokningsomgang-2021.xlsx
- resultat-ansokningsomgang-2022.xlsx
- resultat-ansokningsomgang-2023.xlsx
- resultat-ansokningsomgang-2024.xlsx
- resultat-ansokningsomgang-2025.xlsx


## 5. Source-file understanding

Before designing the curated target table, the notebook first profiles the source workbooks directly.  
This section is deliberately **exploratory but rerunnable**: it turns important source assumptions into visible evidence rather than relying on informal notes.

The profiling below answers six practical questions:
1. Which workbooks and sheets are present?
2. Where are the real headers located in the relevant source tables?
3. Does `Tabell 3` support the intended one-application-per-row main table?
4. Why must `Tabell 4` not be blindly merged into that main table?
5. How much does the `Tabell 3` schema change between 2020 and 2025?
6. Which source-value differences already foreshadow later harmonization work?

### 5.1 Workbook and sheet inventory

The source set is expected to contain one MYH workbook for each application round from **2020** through **2025**.  
This inventory checks both the file set and the internal worksheet layout.

In [4]:
def extract_source_year(file_path: Path) -> int:
    """Extract the MYH application-round year from a workbook filename."""
    match = SOURCE_YEAR_RE.search(file_path.name)
    if match is None:
        raise ValueError(f"Could not extract a source year from: {file_path.name}")
    return int(match.group(1))

source_workbooks = [
    {
        "source_year": extract_source_year(file_path),
        "source_file": file_path.name,
        "file_path": file_path,
    }
    for file_path in raw_excel_files
]

observed_source_years = sorted(workbook["source_year"] for workbook in source_workbooks)
missing_source_years = sorted(set(EXPECTED_SOURCE_YEARS) - set(observed_source_years))
unexpected_source_years = sorted(set(observed_source_years) - set(EXPECTED_SOURCE_YEARS))

if missing_source_years:
    raise FileNotFoundError(
        "Missing expected MYH source workbook(s) for year(s): "
        f"{missing_source_years}."
    )

if unexpected_source_years:
    print(
        "Note: additional Excel workbook years were found outside the planned 2020–2025 scope: "
        f"{unexpected_source_years}. They are visible in the inventory but are not part of the agreed Part 2 scope."
    )

workbook_inventory_rows = []
for workbook in sorted(source_workbooks, key=lambda item: item["source_year"]):
    excel_file = pd.ExcelFile(workbook["file_path"])
    sheet_names = excel_file.sheet_names
    definition_sheet_name = next(
        (sheet for sheet in sheet_names if sheet.startswith("Definitioner")),
        None,
    )
    workbook_inventory_rows.append(
        {
            "source_year": workbook["source_year"],
            "source_file": workbook["source_file"],
            "sheet_count": len(sheet_names),
            "definition_sheet": definition_sheet_name,
            "sheet_names": " | ".join(sheet_names),
        }
    )

workbook_inventory = pd.DataFrame(workbook_inventory_rows).sort_values("source_year").reset_index(drop=True)

display(workbook_inventory)

,source_year,source_file,sheet_count,definition_sheet,sheet_names
0,2020,resultat-ansokningsomgang-2020.xlsx,6,Definitioner och förklaringar,Innehållsförteckning | Definitioner och förkla...
1,2021,resultat-ansokningsomgang-2021.xlsx,6,Definitioner och förklaringar,Innehållsförteckning | Definitioner och förkla...
2,2022,resultat-ansokningsomgang-2022.xlsx,6,Definitioner och förklaringar,Innehållsförteckning | Definitioner och förkla...
3,2023,resultat-ansokningsomgang-2023.xlsx,6,Definitioner och förklaringar,Innehållsförteckning | Definitioner och förkla...
4,2024,resultat-ansokningsomgang-2024.xlsx,6,Definitioner,Innehållsförteckning | Definitioner | Tabell 1...
5,2025,resultat-ansokningsomgang-2025.xlsx,6,Definitioner,Innehållsförteckning | Definitioner | Tabell 1...


### 5.2 Header-row detection for `Tabell 3` and `Tabell 4`

The relevant tables do **not** start on the same Excel row in every year.  
Instead of hard-coding the result as prose only, this profiling step scans the first rows of each table and locates the row containing `Diarienummer`, which is part of the true header row.

The output records both:
- the human-facing Excel row number, and
- the zero-based `header=` index that `pandas.read_excel()` would use later.

In [5]:
PROFILE_SHEETS = ["Tabell 3", "Tabell 4"]
HEADER_SCAN_ROWS = 20
HEADER_MARKER = "Diarienummer"


def detect_header_row(file_path: Path, sheet_name: str, marker: str = HEADER_MARKER) -> int:
    """Return the zero-based row index containing the table header marker."""
    preview = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=None,
        nrows=HEADER_SCAN_ROWS,
    )

    matching_rows = []
    for row_index in range(len(preview)):
        row_values = {
            str(value).strip()
            for value in preview.iloc[row_index].tolist()
            if pd.notna(value)
        }
        if marker in row_values:
            matching_rows.append(row_index)

    if not matching_rows:
        raise ValueError(
            f"Could not find header marker '{marker}' in the first {HEADER_SCAN_ROWS} rows "
            f"of sheet '{sheet_name}' in '{file_path.name}'."
        )

    return matching_rows[0]


header_row_records = []
for workbook in sorted(source_workbooks, key=lambda item: item["source_year"]):
    for sheet_name in PROFILE_SHEETS:
        header_index = detect_header_row(workbook["file_path"], sheet_name)
        header_row_records.append(
            {
                "source_year": workbook["source_year"],
                "source_sheet": sheet_name,
                "excel_header_row": header_index + 1,
                "pandas_header_index": header_index,
            }
        )

header_row_summary = pd.DataFrame(header_row_records).sort_values(
    ["source_sheet", "source_year"]
).reset_index(drop=True)

header_index_lookup = {
    (row.source_year, row.source_sheet): int(row.pandas_header_index)
    for row in header_row_summary.itertuples(index=False)
}

display(header_row_summary)

,source_year,source_sheet,excel_header_row,pandas_header_index
0,2020,Tabell 3,1,0
1,2021,Tabell 3,1,0
2,2022,Tabell 3,1,0
3,2023,Tabell 3,6,5
4,2024,Tabell 3,6,5
5,2025,Tabell 3,7,6
6,2020,Tabell 4,1,0
7,2021,Tabell 4,1,0
8,2022,Tabell 4,1,0
9,2023,Tabell 4,6,5


### 5.3 Read cleaned profiling copies of the source tables

The helper below is **not yet the production ingestion pipeline**.  
Its role in Sub-project 2.2 is narrower: read each profiled source table consistently enough to count rows, inspect identifiers, and compare source schemas.

In [6]:
def read_profile_table(file_path: Path, sheet_name: str, header_index: int) -> pd.DataFrame:
    """Read a source sheet for profiling and remove purely empty rows/columns."""
    table = pd.read_excel(
        file_path,
        sheet_name=sheet_name,
        header=header_index,
    )
    table = table.dropna(how="all").dropna(axis=1, how="all")
    table.columns = [str(column).strip() for column in table.columns]
    return table


tabell_3_tables = {}
tabell_4_tables = {}
for workbook in sorted(source_workbooks, key=lambda item: item["source_year"]):
    year = workbook["source_year"]
    file_path = workbook["file_path"]
    tabell_3_tables[year] = read_profile_table(
        file_path,
        "Tabell 3",
        header_index_lookup[(year, "Tabell 3")],
    )
    tabell_4_tables[year] = read_profile_table(
        file_path,
        "Tabell 4",
        header_index_lookup[(year, "Tabell 4")],
    )

print(f"Profiling copies created for Tabell 3: {sorted(tabell_3_tables)}")
print(f"Profiling copies created for Tabell 4: {sorted(tabell_4_tables)}")

Profiling copies created for Tabell 3: [2020, 2021, 2022, 2023, 2024, 2025]
Profiling copies created for Tabell 4: [2020, 2021, 2022, 2023, 2024, 2025]


### 5.4 `Tabell 3` profile: evidence for the main applications table

The intended main table grain is:
> **one row = one application in one application round**.

For that to be credible, `Tabell 3` should have:
- a clear row count by year,
- no missing `Diarienummer`, and
- no duplicate `Diarienummer` within a year.

In [7]:
tabell_3_profile_rows = []
for year, table in sorted(tabell_3_tables.items()):
    identifiers = table["Diarienummer"]
    tabell_3_profile_rows.append(
        {
            "source_year": year,
            "tabell_3_rows": len(table),
            "tabell_3_columns": len(table.columns),
            "missing_diarienummer": int(identifiers.isna().sum()),
            "duplicate_diarienummer": int(identifiers.duplicated().sum()),
        }
    )

tabell_3_profile = pd.DataFrame(tabell_3_profile_rows).sort_values("source_year").reset_index(drop=True)

display(tabell_3_profile)

assert int(tabell_3_profile["missing_diarienummer"].sum()) == 0, "Unexpected missing Diarienummer in Tabell 3."
assert int(tabell_3_profile["duplicate_diarienummer"].sum()) == 0, "Unexpected duplicate Diarienummer in Tabell 3."

,source_year,tabell_3_rows,tabell_3_columns,missing_diarienummer,duplicate_diarienummer
0,2020,1482,16,0,0
1,2021,1238,17,0,0
2,2022,1207,16,0,0
3,2023,1258,28,0,0
4,2024,1272,28,0,0
5,2025,1184,28,0,0


### 5.5 `Tabell 4` profile: grain warning before any future joins

`Tabell 4` is useful, but it is **not** at the same grain as the planned main applications table.  
The check below compares total rows with the number of unique `Diarienummer` values.  A positive difference shows that at least some applications appear multiple times in `Tabell 4`.

In [8]:
tabell_4_grain_rows = []
for year, table in sorted(tabell_4_tables.items()):
    unique_application_ids = int(table["Diarienummer"].nunique(dropna=True))
    tabell_4_grain_rows.append(
        {
            "source_year": year,
            "tabell_4_rows": len(table),
            "unique_diarienummer": unique_application_ids,
            "rows_above_unique_application_count": len(table) - unique_application_ids,
        }
    )

tabell_4_grain_check = pd.DataFrame(tabell_4_grain_rows).sort_values("source_year").reset_index(drop=True)

display(tabell_4_grain_check)

assert (
    tabell_4_grain_check["rows_above_unique_application_count"] > 0
).all(), "Expected Tabell 4 to show repeated application identifiers in every profiled year."

,source_year,tabell_4_rows,unique_diarienummer,rows_above_unique_application_count
0,2020,1903,1482,421
1,2021,1586,1238,348
2,2022,1621,1207,414
3,2023,642,243,399
4,2024,842,292,550
5,2025,836,295,541


### 5.6 `Tabell 3` schema comparison across 2020–2025

A cross-year curated dataset cannot assume that every source year has the same input structure.  
This comparison records:
- the column count by year, and
- which source columns appear in which years.

The result gives concrete evidence for the later harmonization/specification work in Sub-project 2.3.

In [9]:
tabell_3_schema_by_year = {
    year: list(table.columns)
    for year, table in sorted(tabell_3_tables.items())
}

schema_count_summary = pd.DataFrame(
    [
        {
            "source_year": year,
            "tabell_3_columns": len(columns),
            "column_names": " | ".join(columns),
        }
        for year, columns in tabell_3_schema_by_year.items()
    ]
).sort_values("source_year").reset_index(drop=True)

all_tabell_3_columns = sorted(
    set().union(*(set(columns) for columns in tabell_3_schema_by_year.values()))
)

schema_presence_rows = []
for column_name in all_tabell_3_columns:
    years_present = [
        year
        for year, columns in tabell_3_schema_by_year.items()
        if column_name in columns
    ]
    schema_presence_rows.append(
        {
            "column_name": column_name,
            "years_present": ", ".join(str(year) for year in years_present),
            "present_in_year_count": len(years_present),
        }
    )

schema_presence_summary = pd.DataFrame(schema_presence_rows).sort_values(
    ["present_in_year_count", "column_name"],
    ascending=[False, True],
).reset_index(drop=True)

schema_variation_summary = schema_presence_summary[
    schema_presence_summary["present_in_year_count"] < len(EXPECTED_SOURCE_YEARS)
].reset_index(drop=True)

display(schema_count_summary)
display(schema_variation_summary)

,source_year,tabell_3_columns,column_names
0,2020,16,Utbildningsområde | Utbildningsnamn | Län | Ko...
1,2021,17,Utbildningsområde | Utbildningsnamn | Län | Ko...
2,2022,16,Utbildningsområde | Utbildningsnamn | Beslut |...
3,2023,28,Utbildningsområde | SUN5 inriktning | SUN5 inr...
4,2024,28,Utbildningsområde | SUN5 inriktning | SUN5 inr...
5,2025,28,Utbildningsområde | SUN5 inriktning | SUN5 inr...


,column_name,years_present,present_in_year_count
0,Flera kommuner,"2022, 2023, 2024, 2025",4
1,Typ av examen,"2021, 2022, 2023, 2024",4
2,Beviljade platser totalt,"2023, 2024, 2025",3
3,Beviljade platser utbildningsomgång 1,"2023, 2024, 2025",3
4,Beviljade platser utbildningsomgång 2,"2023, 2024, 2025",3
5,Beviljade platser utbildningsomgång 3,"2023, 2024, 2025",3
6,Beviljade platser utbildningsomgång 4,"2023, 2024, 2025",3
7,Beviljade platser utbildningsomgång 5,"2023, 2024, 2025",3
8,SUN5 inriktning,"2023, 2024, 2025",3
9,SUN5 inriktning namn,"2023, 2024, 2025",3


### 5.7 Source values that already signal later harmonization needs

This is still source exploration, not cleaning.  
However, two columns already show year-to-year vocabulary differences that will matter later:
- `Beslut`, where rejection wording changes and 2025 adds `Återkallad`,
- `Huvudmannatyp`, where `Landsting` is replaced by `Region` in later years.

In [10]:
def summarize_distinct_values(tables_by_year: dict[int, pd.DataFrame], column_name: str) -> pd.DataFrame:
    """Summarize sorted distinct non-null source values for one column by source year."""
    rows = []
    for year, table in sorted(tables_by_year.items()):
        distinct_values = sorted(str(value) for value in table[column_name].dropna().unique())
        rows.append(
            {
                "source_year": year,
                "source_column": column_name,
                "distinct_values": " | ".join(distinct_values),
            }
        )
    return pd.DataFrame(rows)

beslut_value_summary = summarize_distinct_values(tabell_3_tables, "Beslut")
huvudmannatyp_value_summary = summarize_distinct_values(tabell_3_tables, "Huvudmannatyp")

display(beslut_value_summary)
display(huvudmannatyp_value_summary)

,source_year,source_column,distinct_values
0,2020,Beslut,Beviljad | Ej beviljad
1,2021,Beslut,Beviljad | Ej beviljad
2,2022,Beslut,Avslag | Beviljad
3,2023,Beslut,Avslag | Beviljad
4,2024,Beslut,Avslag | Beviljad
5,2025,Beslut,Avslag | Beviljad | Återkallad


,source_year,source_column,distinct_values
0,2020,Huvudmannatyp,Kommun | Landsting | Privat | Statlig
1,2021,Huvudmannatyp,Kommun | Landsting | Privat | Statlig
2,2022,Huvudmannatyp,Kommun | Privat | Region | Statlig
3,2023,Huvudmannatyp,Kommun | Privat | Region | Statlig
4,2024,Huvudmannatyp,Kommun | Privat | Region | Statlig
5,2025,Huvudmannatyp,Kommun | Privat | Region


## 6. Why `Tabell 3` is the main source and `Tabell 4` is not merged

The profiling evidence above supports the project’s current data-grain decision:

1. **`Tabell 3` fits the planned main table grain.**  
   Across all six source years, `Tabell 3` has one row per recorded application, zero missing `Diarienummer`, and zero duplicate `Diarienummer` values within each year.

2. **`Tabell 4` is at a more detailed grain.**  
   In every year, `Tabell 4` has more rows than unique `Diarienummer` values. That means some applications appear multiple times there.

3. **Therefore, `Tabell 4` must not be blindly joined into the main applications table.**  
   A direct merge would risk duplicating applications and corrupting counts. If `Tabell 4` is ever used later, it should be handled as a separate more-detailed table or joined only after an explicit aggregation/design decision.

This is a **grain conclusion**, not a final schema conclusion. The formal target schema and source-to-target harmonization rules belong to Sub-project **2.3**.

## 7. Target schema and harmonization specification

Sub-project **2.3** now locks the **main curated-table design** before production ingestion is implemented.

### 7.0 Schema-design decisions now fixed
- The curated dataset remains a **single `Tabell 3`-based applications table**.
- The grain remains: **one row = one application in one application round**.
- Curated field names use lowercase `snake_case` without Swedish diacritics where practical.
- Raw categorical fields are retained when useful, and normalized companion fields are added where cross-year harmonization is required.
- Fields introduced only in later workbooks are **kept when they add clear analytical value**, and they will be null for earlier years where the source column does not exist.
- Source traceability is preserved through:
  - `source_year`
  - `source_file`
  - `source_sheet`
  - `source_row`

### 7.0.1 Traceability semantics
The final curated table will use the following meanings:

| Curated field | Design rule |
|---|---|
| `source_year` | MYH application-round year extracted from the workbook filename |
| `source_file` | Source workbook filename only, not an absolute local path |
| `source_sheet` | Expected to be `Tabell 3` for the main table |
| `source_row` | **1-based Excel row number** of the original source record in `Tabell 3` |

`source_row` is intentionally defined as an Excel row reference rather than a temporary DataFrame index.  
The later ingestion implementation can calculate it from the detected header position and row offset, preserving a direct audit path back to the raw workbook.

### 7.0.2 Structural-null policy
Some fields are valuable enough to retain even though they appear only in later source years.  
For those fields, earlier years will carry **structural nulls** during ingestion:

- not fabricated values,
- not silent zeroes,
- not treated as accidental data loss.

This policy keeps one stable target schema across 2020–2025 while preserving the historical limits of the raw sources.


In [11]:
CURATED_SCHEMA_FIELDS = [
    {
        "column_order": 1,
        "curated_field": "source_year",
        "field_group": "traceability",
        "source_basis": "workbook filename / import metadata",
        "availability": "all years",
        "implementation_note": "MYH application-round year",
    },
    {
        "column_order": 2,
        "curated_field": "source_file",
        "field_group": "traceability",
        "source_basis": "workbook filename",
        "availability": "all years",
        "implementation_note": "Portable filename only, not an absolute path",
    },
    {
        "column_order": 3,
        "curated_field": "source_sheet",
        "field_group": "traceability",
        "source_basis": "import metadata",
        "availability": "all years",
        "implementation_note": "Expected to be Tabell 3",
    },
    {
        "column_order": 4,
        "curated_field": "source_row",
        "field_group": "traceability",
        "source_basis": "detected header row + source-record offset",
        "availability": "all years",
        "implementation_note": "1-based Excel row number of the original Tabell 3 record",
    },
    {
        "column_order": 5,
        "curated_field": "diarienummer",
        "field_group": "application_identity",
        "source_basis": "Diarienummer",
        "availability": "all years",
        "implementation_note": "Main application identifier within source year",
    },
    {
        "column_order": 6,
        "curated_field": "utbildningsnamn",
        "field_group": "application_identity",
        "source_basis": "Utbildningsnamn",
        "availability": "all years",
        "implementation_note": "Program name",
    },
    {
        "column_order": 7,
        "curated_field": "utbildningsomrade",
        "field_group": "application_identity",
        "source_basis": "Utbildningsområde",
        "availability": "all years",
        "implementation_note": "Broad education area",
    },
    {
        "column_order": 8,
        "curated_field": "beslut",
        "field_group": "decision",
        "source_basis": "Beslut",
        "availability": "all years",
        "implementation_note": "Original source decision value retained",
    },
    {
        "column_order": 9,
        "curated_field": "beslut_normalized",
        "field_group": "decision",
        "source_basis": "derived from beslut",
        "availability": "all years",
        "implementation_note": "Cross-year decision category",
    },
    {
        "column_order": 10,
        "curated_field": "is_approved",
        "field_group": "decision",
        "source_basis": "derived from beslut_normalized",
        "availability": "all years",
        "implementation_note": "True only when beslut_normalized == 'approved'",
    },
    {
        "column_order": 11,
        "curated_field": "lan",
        "field_group": "geography",
        "source_basis": "Län",
        "availability": "all years",
        "implementation_note": "County recorded in Tabell 3",
    },
    {
        "column_order": 12,
        "curated_field": "kommun",
        "field_group": "geography",
        "source_basis": "Kommun",
        "availability": "all years",
        "implementation_note": "Municipality recorded in Tabell 3",
    },
    {
        "column_order": 13,
        "curated_field": "flera_kommuner",
        "field_group": "geography",
        "source_basis": "Flera studiekommuner / Flera kommuner",
        "availability": "all years",
        "implementation_note": "Source yes/no indicator under a harmonized target name",
    },
    {
        "column_order": 14,
        "curated_field": "has_multiple_municipalities",
        "field_group": "geography",
        "source_basis": "derived from flera_kommuner",
        "availability": "all years",
        "implementation_note": "Boolean convenience field; Ja -> True, Nej -> False",
    },
    {
        "column_order": 15,
        "curated_field": "antal_kommuner",
        "field_group": "geography",
        "source_basis": "Antal kommuner",
        "availability": "all years",
        "implementation_note": "Number of municipalities",
    },
    {
        "column_order": 16,
        "curated_field": "yh_poang",
        "field_group": "program_structure",
        "source_basis": "YH-poäng",
        "availability": "all years",
        "implementation_note": "Program extent in YH credits",
    },
    {
        "column_order": 17,
        "curated_field": "studieform",
        "field_group": "program_structure",
        "source_basis": "Studieform",
        "availability": "all years",
        "implementation_note": "Original source delivery form",
    },
    {
        "column_order": 18,
        "curated_field": "is_distance_based",
        "field_group": "program_structure",
        "source_basis": "derived from studieform",
        "availability": "all years",
        "implementation_note": "Boolean convenience field; Distans -> True, Bunden -> False",
    },
    {
        "column_order": 19,
        "curated_field": "studietakt_procent",
        "field_group": "program_structure",
        "source_basis": "Studietakt %",
        "availability": "all years",
        "implementation_note": "Study pace as a percentage",
    },
    {
        "column_order": 20,
        "curated_field": "examenstyp",
        "field_group": "program_structure",
        "source_basis": "Typ av examen / Examenstyp",
        "availability": "2021-2025; structural null in 2020",
        "implementation_note": "Harmonized target name for the exam-type field",
    },
    {
        "column_order": 21,
        "curated_field": "utbildningsanordnare",
        "field_group": "provider",
        "source_basis": "Utbildningsanordnare administrativ enhet",
        "availability": "all years",
        "implementation_note": "Provider / administrative unit",
    },
    {
        "column_order": 22,
        "curated_field": "huvudmannatyp",
        "field_group": "provider",
        "source_basis": "Huvudmannatyp",
        "availability": "all years",
        "implementation_note": "Original source provider-type value retained",
    },
    {
        "column_order": 23,
        "curated_field": "huvudmannatyp_normalized",
        "field_group": "provider",
        "source_basis": "derived from huvudmannatyp",
        "availability": "all years",
        "implementation_note": "Landsting and Region harmonized to Region",
    },
    {
        "column_order": 24,
        "curated_field": "sokta_utbildningsomgangar",
        "field_group": "application_scope",
        "source_basis": "Sökta utbildningsomgångar",
        "availability": "all years",
        "implementation_note": "Requested education rounds",
    },
    {
        "column_order": 25,
        "curated_field": "beviljade_utbildningsomgangar",
        "field_group": "application_scope",
        "source_basis": "Beviljade utbildningsomgångar",
        "availability": "all years",
        "implementation_note": "Approved education rounds",
    },
    {
        "column_order": 26,
        "curated_field": "sun5_inriktning",
        "field_group": "newer_classification",
        "source_basis": "SUN5 inriktning",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Retained because it adds useful classification detail",
    },
    {
        "column_order": 27,
        "curated_field": "sun5_inriktning_namn",
        "field_group": "newer_classification",
        "source_basis": "SUN5 inriktning namn",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Human-readable SUN5 label",
    },
    {
        "column_order": 28,
        "curated_field": "seqf_niva",
        "field_group": "newer_classification",
        "source_basis": "SeQF nivå",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Retained as a later-year classification field",
    },
    {
        "column_order": 29,
        "curated_field": "smalt_yrkesomrade",
        "field_group": "newer_classification",
        "source_basis": "Smalt yrkesområde",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Retained as a later-year classification field",
    },
    {
        "column_order": 30,
        "curated_field": "sokta_platser_per_utbildningsomgang",
        "field_group": "newer_seat_totals",
        "source_basis": "Sökta platser per utbildningsomgång",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Retained as a compact capacity-related summary",
    },
    {
        "column_order": 31,
        "curated_field": "sokta_platser_totalt",
        "field_group": "newer_seat_totals",
        "source_basis": "Sökta platser totalt",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Retained as a compact capacity-related summary",
    },
    {
        "column_order": 32,
        "curated_field": "beviljade_platser_totalt",
        "field_group": "newer_seat_totals",
        "source_basis": "Beviljade platser totalt",
        "availability": "2023-2025; structural null in 2020-2022",
        "implementation_note": "Retained as a compact capacity-related summary",
    },
]

curated_schema_spec = pd.DataFrame(CURATED_SCHEMA_FIELDS).sort_values("column_order").reset_index(drop=True)

display(curated_schema_spec)

,column_order,curated_field,field_group,source_basis,availability,implementation_note
0,1,source_year,traceability,workbook filename / import metadata,all years,MYH application-round year
1,2,source_file,traceability,workbook filename,all years,"Portable filename only, not an absolute path"
2,3,source_sheet,traceability,import metadata,all years,Expected to be Tabell 3
3,4,source_row,traceability,detected header row + source-record offset,all years,1-based Excel row number of the original Tabel...
4,5,diarienummer,application_identity,Diarienummer,all years,Main application identifier within source year
5,6,utbildningsnamn,application_identity,Utbildningsnamn,all years,Program name
6,7,utbildningsomrade,application_identity,Utbildningsområde,all years,Broad education area
7,8,beslut,decision,Beslut,all years,Original source decision value retained
8,9,beslut_normalized,decision,derived from beslut,all years,Cross-year decision category
9,10,is_approved,decision,derived from beslut_normalized,all years,True only when beslut_normalized == 'approved'


### 7.1 Source-to-target mapping table

The table below formalizes how the final curated fields connect back to the observed `Tabell 3` source columns.  
It is intentionally a **design specification**, not yet the reusable ingestion pipeline. The production functions that apply these rules belong to Sub-project **2.4**.


In [12]:
SOURCE_TO_TARGET_MAPPING = [
    {"curated_field": "source_year", "source_column_or_rule": "import metadata from workbook filename", "source_years": "2020-2025", "harmonization_rule": "Extract the application-round year"},
    {"curated_field": "source_file", "source_column_or_rule": "workbook filename", "source_years": "2020-2025", "harmonization_rule": "Store filename only"},
    {"curated_field": "source_sheet", "source_column_or_rule": "sheet metadata", "source_years": "2020-2025", "harmonization_rule": "Store 'Tabell 3'"},
    {"curated_field": "source_row", "source_column_or_rule": "detected Excel source row", "source_years": "2020-2025", "harmonization_rule": "Store 1-based Excel row number"},
    {"curated_field": "diarienummer", "source_column_or_rule": "Diarienummer", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "utbildningsnamn", "source_column_or_rule": "Utbildningsnamn", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "utbildningsomrade", "source_column_or_rule": "Utbildningsområde", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "beslut", "source_column_or_rule": "Beslut", "source_years": "2020-2025", "harmonization_rule": "Retain original source value"},
    {"curated_field": "beslut_normalized", "source_column_or_rule": "derived from beslut", "source_years": "2020-2025", "harmonization_rule": "Beviljad -> approved; Ej beviljad/Avslag -> rejected; Återkallad -> withdrawn"},
    {"curated_field": "is_approved", "source_column_or_rule": "derived from beslut_normalized", "source_years": "2020-2025", "harmonization_rule": "approved -> True; otherwise False"},
    {"curated_field": "lan", "source_column_or_rule": "Län", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "kommun", "source_column_or_rule": "Kommun", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "flera_kommuner", "source_column_or_rule": "2020-2021: Flera studiekommuner; 2022-2025: Flera kommuner", "source_years": "2020-2025", "harmonization_rule": "Map both source names into one target field"},
    {"curated_field": "has_multiple_municipalities", "source_column_or_rule": "derived from flera_kommuner", "source_years": "2020-2025", "harmonization_rule": "Ja -> True; Nej -> False"},
    {"curated_field": "antal_kommuner", "source_column_or_rule": "Antal kommuner", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "yh_poang", "source_column_or_rule": "YH-poäng", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "studieform", "source_column_or_rule": "Studieform", "source_years": "2020-2025", "harmonization_rule": "Retain original source value"},
    {"curated_field": "is_distance_based", "source_column_or_rule": "derived from studieform", "source_years": "2020-2025", "harmonization_rule": "Distans -> True; Bunden -> False"},
    {"curated_field": "studietakt_procent", "source_column_or_rule": "Studietakt %", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "examenstyp", "source_column_or_rule": "2021-2024: Typ av examen; 2025: Examenstyp", "source_years": "2021-2025", "harmonization_rule": "Map both source names into one target field; structural null in 2020"},
    {"curated_field": "utbildningsanordnare", "source_column_or_rule": "Utbildningsanordnare administrativ enhet", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "huvudmannatyp", "source_column_or_rule": "Huvudmannatyp", "source_years": "2020-2025", "harmonization_rule": "Retain original source value"},
    {"curated_field": "huvudmannatyp_normalized", "source_column_or_rule": "derived from huvudmannatyp", "source_years": "2020-2025", "harmonization_rule": "Landsting/Region -> Region; other categories retained"},
    {"curated_field": "sokta_utbildningsomgangar", "source_column_or_rule": "Sökta utbildningsomgångar", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "beviljade_utbildningsomgangar", "source_column_or_rule": "Beviljade utbildningsomgångar", "source_years": "2020-2025", "harmonization_rule": "Rename only"},
    {"curated_field": "sun5_inriktning", "source_column_or_rule": "SUN5 inriktning", "source_years": "2023-2025", "harmonization_rule": "Retain; structural null in 2020-2022"},
    {"curated_field": "sun5_inriktning_namn", "source_column_or_rule": "SUN5 inriktning namn", "source_years": "2023-2025", "harmonization_rule": "Retain; structural null in 2020-2022"},
    {"curated_field": "seqf_niva", "source_column_or_rule": "SeQF nivå", "source_years": "2023-2025", "harmonization_rule": "Retain; structural null in 2020-2022"},
    {"curated_field": "smalt_yrkesomrade", "source_column_or_rule": "Smalt yrkesområde", "source_years": "2023-2025", "harmonization_rule": "Retain; structural null in 2020-2022"},
    {"curated_field": "sokta_platser_per_utbildningsomgang", "source_column_or_rule": "Sökta platser per utbildningsomgång", "source_years": "2023-2025", "harmonization_rule": "Retain summary field; structural null in 2020-2022"},
    {"curated_field": "sokta_platser_totalt", "source_column_or_rule": "Sökta platser totalt", "source_years": "2023-2025", "harmonization_rule": "Retain summary field; structural null in 2020-2022"},
    {"curated_field": "beviljade_platser_totalt", "source_column_or_rule": "Beviljade platser totalt", "source_years": "2023-2025", "harmonization_rule": "Retain summary field; structural null in 2020-2022"},
]

source_to_target_mapping = pd.DataFrame(SOURCE_TO_TARGET_MAPPING)

assert list(source_to_target_mapping["curated_field"]) == list(curated_schema_spec["curated_field"]), (
    "The source-to-target mapping should match the locked curated schema order."
)

display(source_to_target_mapping)

,curated_field,source_column_or_rule,source_years,harmonization_rule
0,source_year,import metadata from workbook filename,2020-2025,Extract the application-round year
1,source_file,workbook filename,2020-2025,Store filename only
2,source_sheet,sheet metadata,2020-2025,Store 'Tabell 3'
3,source_row,detected Excel source row,2020-2025,Store 1-based Excel row number
4,diarienummer,Diarienummer,2020-2025,Rename only
5,utbildningsnamn,Utbildningsnamn,2020-2025,Rename only
6,utbildningsomrade,Utbildningsområde,2020-2025,Rename only
7,beslut,Beslut,2020-2025,Retain original source value
8,beslut_normalized,derived from beslut,2020-2025,Beviljad -> approved; Ej beviljad/Avslag -> re...
9,is_approved,derived from beslut_normalized,2020-2025,approved -> True; otherwise False


### 7.2 Inclusion and exclusion decisions for observed `Tabell 3` columns

A good target schema should not silently drop source fields or silently include every field without a reason.  
The table below accounts for **every observed `Tabell 3` source column** and records whether it is:
- retained directly,
- harmonized into a target field,
- or excluded from the main curated applications table.


In [13]:
SOURCE_COLUMN_DISPOSITION = [
    {"source_column": "Utbildningsområde", "design_status": "include", "curated_target": "utbildningsomrade", "reason": "Core cross-year program descriptor"},
    {"source_column": "Utbildningsnamn", "design_status": "include", "curated_target": "utbildningsnamn", "reason": "Core application/program descriptor"},
    {"source_column": "Län", "design_status": "include", "curated_target": "lan", "reason": "Core geography field"},
    {"source_column": "Kommun", "design_status": "include", "curated_target": "kommun", "reason": "Core geography field"},
    {"source_column": "Flera studiekommuner", "design_status": "harmonize", "curated_target": "flera_kommuner", "reason": "Older source name for the same multi-municipality concept"},
    {"source_column": "Flera kommuner", "design_status": "harmonize", "curated_target": "flera_kommuner", "reason": "Later source name for the same multi-municipality concept"},
    {"source_column": "Antal kommuner", "design_status": "include", "curated_target": "antal_kommuner", "reason": "Useful count field with stable meaning"},
    {"source_column": "Antal län", "design_status": "exclude", "curated_target": "", "reason": "Only present in 2020 and not needed for the chosen main-table design"},
    {"source_column": "YH-poäng", "design_status": "include", "curated_target": "yh_poang", "reason": "Core program-structure measure"},
    {"source_column": "Studieform", "design_status": "include", "curated_target": "studieform", "reason": "Core program-structure category"},
    {"source_column": "Studietakt %", "design_status": "include", "curated_target": "studietakt_procent", "reason": "Core program-structure measure"},
    {"source_column": "Typ av examen", "design_status": "harmonize", "curated_target": "examenstyp", "reason": "2021-2024 source name for exam type"},
    {"source_column": "Examenstyp", "design_status": "harmonize", "curated_target": "examenstyp", "reason": "2025 source name for exam type"},
    {"source_column": "Utbildningsanordnare administrativ enhet", "design_status": "include", "curated_target": "utbildningsanordnare", "reason": "Core provider descriptor"},
    {"source_column": "Huvudmannatyp", "design_status": "include", "curated_target": "huvudmannatyp", "reason": "Raw provider-type category retained before normalization"},
    {"source_column": "Sökta utbildningsomgångar", "design_status": "include", "curated_target": "sokta_utbildningsomgangar", "reason": "Cross-year application-scope field"},
    {"source_column": "Beviljade utbildningsomgångar", "design_status": "include", "curated_target": "beviljade_utbildningsomgangar", "reason": "Cross-year application-scope field"},
    {"source_column": "Diarienummer", "design_status": "include", "curated_target": "diarienummer", "reason": "Application identifier"},
    {"source_column": "Beslut", "design_status": "include", "curated_target": "beslut", "reason": "Raw decision value retained before normalization"},
    {"source_column": "SUN5 inriktning", "design_status": "include", "curated_target": "sun5_inriktning", "reason": "Useful 2023-2025 classification field; earlier years structurally null"},
    {"source_column": "SUN5 inriktning namn", "design_status": "include", "curated_target": "sun5_inriktning_namn", "reason": "Useful 2023-2025 classification label; earlier years structurally null"},
    {"source_column": "SeQF nivå", "design_status": "include", "curated_target": "seqf_niva", "reason": "Useful 2023-2025 classification field; earlier years structurally null"},
    {"source_column": "Smalt yrkesområde", "design_status": "include", "curated_target": "smalt_yrkesomrade", "reason": "Useful 2023-2025 classification field; earlier years structurally null"},
    {"source_column": "Sökta platser per utbildningsomgång", "design_status": "include", "curated_target": "sokta_platser_per_utbildningsomgang", "reason": "Compact 2023-2025 seat-demand summary; earlier years structurally null"},
    {"source_column": "Sökta platser totalt", "design_status": "include", "curated_target": "sokta_platser_totalt", "reason": "Compact 2023-2025 seat-demand summary; earlier years structurally null"},
    {"source_column": "Beviljade platser totalt", "design_status": "include", "curated_target": "beviljade_platser_totalt", "reason": "Compact 2023-2025 seat-approval summary; earlier years structurally null"},
    {"source_column": "Beviljade platser utbildningsomgång 1", "design_status": "exclude", "curated_target": "", "reason": "Round-specific repeating measure; too wide/sparse for the chosen main table"},
    {"source_column": "Beviljade platser utbildningsomgång 2", "design_status": "exclude", "curated_target": "", "reason": "Round-specific repeating measure; too wide/sparse for the chosen main table"},
    {"source_column": "Beviljade platser utbildningsomgång 3", "design_status": "exclude", "curated_target": "", "reason": "Round-specific repeating measure; too wide/sparse for the chosen main table"},
    {"source_column": "Beviljade platser utbildningsomgång 4", "design_status": "exclude", "curated_target": "", "reason": "Round-specific repeating measure; too wide/sparse for the chosen main table"},
    {"source_column": "Beviljade platser utbildningsomgång 5", "design_status": "exclude", "curated_target": "", "reason": "Round-specific repeating measure; too wide/sparse for the chosen main table"},
]

source_column_disposition = pd.DataFrame(SOURCE_COLUMN_DISPOSITION).sort_values(
    ["design_status", "source_column"]
).reset_index(drop=True)

observed_tabell_3_source_columns = set(all_tabell_3_columns)
accounted_source_columns = set(source_column_disposition["source_column"])

assert accounted_source_columns == observed_tabell_3_source_columns, (
    "Every observed Tabell 3 source column must be explicitly accounted for in the design disposition."
)

display(source_column_disposition)

,source_column,design_status,curated_target,reason
0,Antal län,exclude,,Only present in 2020 and not needed for the ch...
1,Beviljade platser utbildningsomgång 1,exclude,,Round-specific repeating measure; too wide/spa...
2,Beviljade platser utbildningsomgång 2,exclude,,Round-specific repeating measure; too wide/spa...
3,Beviljade platser utbildningsomgång 3,exclude,,Round-specific repeating measure; too wide/spa...
4,Beviljade platser utbildningsomgång 4,exclude,,Round-specific repeating measure; too wide/spa...
5,Beviljade platser utbildningsomgång 5,exclude,,Round-specific repeating measure; too wide/spa...
6,Examenstyp,harmonize,examenstyp,2025 source name for exam type
7,Flera kommuner,harmonize,flera_kommuner,Later source name for the same multi-municipal...
8,Flera studiekommuner,harmonize,flera_kommuner,Older source name for the same multi-municipal...
9,Typ av examen,harmonize,examenstyp,2021-2024 source name for exam type


### 7.3 Normalized-value specifications and design-level coverage checks

Two cross-year categorical harmonizations are formally locked here:

1. `Beslut` → `beslut_normalized`
2. `Huvudmannatyp` → `huvudmannatyp_normalized`

The mappings are validated against the distinct source values already observed in the six profiled workbooks.  
This remains a **design-level coverage check**; the reusable transformation code will be implemented in Sub-project **2.4**.


In [14]:
BESLUT_NORMALIZATION = {
    "Beviljad": "approved",
    "Ej beviljad": "rejected",
    "Avslag": "rejected",
    "Återkallad": "withdrawn",
}

HUVUDMANNATYP_NORMALIZATION = {
    "Landsting": "Region",
    "Region": "Region",
    "Kommun": "Kommun",
    "Privat": "Privat",
    "Statlig": "Statlig",
}

beslut_normalization_spec = pd.DataFrame(
    [
        {"source_value": source_value, "normalized_value": normalized_value}
        for source_value, normalized_value in BESLUT_NORMALIZATION.items()
    ]
).sort_values("source_value").reset_index(drop=True)

huvudmannatyp_normalization_spec = pd.DataFrame(
    [
        {"source_value": source_value, "normalized_value": normalized_value}
        for source_value, normalized_value in HUVUDMANNATYP_NORMALIZATION.items()
    ]
).sort_values("source_value").reset_index(drop=True)

observed_beslut_values = {
    str(value).strip()
    for table in tabell_3_tables.values()
    for value in table["Beslut"].dropna().unique()
}
observed_huvudmannatyp_values = {
    str(value).strip()
    for table in tabell_3_tables.values()
    for value in table["Huvudmannatyp"].dropna().unique()
}

unmapped_beslut_values = sorted(observed_beslut_values - set(BESLUT_NORMALIZATION))
unmapped_huvudmannatyp_values = sorted(
    observed_huvudmannatyp_values - set(HUVUDMANNATYP_NORMALIZATION)
)

normalization_coverage_summary = pd.DataFrame(
    [
        {
            "field": "Beslut -> beslut_normalized",
            "observed_source_values": " | ".join(sorted(observed_beslut_values)),
            "unmapped_source_values": " | ".join(unmapped_beslut_values) if unmapped_beslut_values else "None",
        },
        {
            "field": "Huvudmannatyp -> huvudmannatyp_normalized",
            "observed_source_values": " | ".join(sorted(observed_huvudmannatyp_values)),
            "unmapped_source_values": " | ".join(unmapped_huvudmannatyp_values) if unmapped_huvudmannatyp_values else "None",
        },
    ]
)

display(beslut_normalization_spec)
display(huvudmannatyp_normalization_spec)
display(normalization_coverage_summary)

assert not unmapped_beslut_values, "Observed Beslut value(s) are missing from BESLUT_NORMALIZATION."
assert not unmapped_huvudmannatyp_values, "Observed Huvudmannatyp value(s) are missing from HUVUDMANNATYP_NORMALIZATION."

,source_value,normalized_value
0,Avslag,rejected
1,Beviljad,approved
2,Ej beviljad,rejected
3,Återkallad,withdrawn


,source_value,normalized_value
0,Kommun,Kommun
1,Landsting,Region
2,Privat,Privat
3,Region,Region
4,Statlig,Statlig


,field,observed_source_values,unmapped_source_values
0,Beslut -> beslut_normalized,Avslag | Beviljad | Ej beviljad | Återkallad,None
1,Huvudmannatyp -> huvudmannatyp_normalized,Kommun | Landsting | Privat | Region | Statlig,None


### 7.4 Schema conclusion

The target design is now sufficiently specific for implementation work to begin in Sub-project **2.4**:

- the **final curated field set and order** are fixed,
- each target field has a documented source basis,
- every observed `Tabell 3` source column has an explicit keep / harmonize / exclude decision,
- later-year fields have an explicit structural-null policy,
- and the two main categorical normalization rules already have design-level coverage checks.

The next stage should convert this specification into a reusable ingestion-and-standardization pipeline that creates consistent, year-specific intermediate tables before later cleaning and enrichment.


## 8. Reusable ingestion and standardization *(Sub-project 2.4)*

The 2.3 design layer now becomes executable import logic. This section keeps import mechanics separate from later cleaning and enrichment: first record the year-specific `Tabell 3` layout, then use one reader function instead of repeated ad hoc `read_excel(...)` calls, and then standardize/concatenate in the next subsection.

### 8.1 Year-aware `Tabell 3` import configuration and reusable reader

The real `Tabell 3` header row differs across the six MYH workbooks. A compact configuration table makes that structural difference explicit, reviewable, and rerunnable. The first version of the reader below uses the configuration, removes purely empty rows/columns, and preserves portable traceability fields, including the original 1-based Excel source row.

In [15]:
TABELL_3_HEADER_ROWS = {
    2020: {"excel_header_row": 1, "pandas_header_index": 0},
    2021: {"excel_header_row": 1, "pandas_header_index": 0},
    2022: {"excel_header_row": 1, "pandas_header_index": 0},
    2023: {"excel_header_row": 6, "pandas_header_index": 5},
    2024: {"excel_header_row": 6, "pandas_header_index": 5},
    2025: {"excel_header_row": 7, "pandas_header_index": 6},
}

source_workbook_by_year = {
    int(workbook["source_year"]): workbook
    for workbook in source_workbooks
}

source_import_config = pd.DataFrame(
    [
        {
            "source_year": year,
            "source_file": source_workbook_by_year[year]["source_file"],
            "source_sheet": "Tabell 3",
            "excel_header_row": header_metadata["excel_header_row"],
            "pandas_header_index": header_metadata["pandas_header_index"],
        }
        for year, header_metadata in TABELL_3_HEADER_ROWS.items()
    ]
).sort_values("source_year").reset_index(drop=True)

assert list(source_import_config["source_year"]) == EXPECTED_SOURCE_YEARS, (
    "The Tabell 3 source import configuration must cover the agreed 2020–2025 scope."
)
assert source_import_config["source_file"].is_unique, (
    "Each configured source year should resolve to one portable workbook filename."
)

display(source_import_config)

LOCKED_CURATED_SCHEMA = list(curated_schema_spec["curated_field"])

SOURCE_TO_STANDARDIZED_COLUMN = {
    "Diarienummer": "diarienummer",
    "Utbildningsnamn": "utbildningsnamn",
    "Utbildningsområde": "utbildningsomrade",
    "Beslut": "beslut",
    "Län": "lan",
    "Kommun": "kommun",
    "Flera studiekommuner": "flera_kommuner",
    "Flera kommuner": "flera_kommuner",
    "Antal kommuner": "antal_kommuner",
    "YH-poäng": "yh_poang",
    "Studieform": "studieform",
    "Studietakt %": "studietakt_procent",
    "Typ av examen": "examenstyp",
    "Examenstyp": "examenstyp",
    "Utbildningsanordnare administrativ enhet": "utbildningsanordnare",
    "Huvudmannatyp": "huvudmannatyp",
    "Sökta utbildningsomgångar": "sokta_utbildningsomgangar",
    "Beviljade utbildningsomgångar": "beviljade_utbildningsomgangar",
    "SUN5 inriktning": "sun5_inriktning",
    "SUN5 inriktning namn": "sun5_inriktning_namn",
    "SeQF nivå": "seqf_niva",
    "Smalt yrkesområde": "smalt_yrkesomrade",
    "Sökta platser per utbildningsomgång": "sokta_platser_per_utbildningsomgang",
    "Sökta platser totalt": "sokta_platser_totalt",
    "Beviljade platser totalt": "beviljade_platser_totalt",
}

INGESTION_STAGE_PLACEHOLDER_FIELDS = [
    "beslut_normalized",
    "is_approved",
    "has_multiple_municipalities",
    "is_distance_based",
    "huvudmannatyp_normalized",
]

CONCAT_STABLE_NULLABLE_INTEGER_FIELDS = [
    "seqf_niva",
    "sokta_platser_per_utbildningsomgang",
    "sokta_platser_totalt",
    "beviljade_platser_totalt",
]


def read_standardized_tabell_3(config: dict[str, object]) -> pd.DataFrame:
    """Read one configured Tabell 3 sheet and align it to the locked 32-field schema."""
    file_path = RAW_DATA_DIR / str(config["source_file"])
    source_table = pd.read_excel(
        file_path,
        sheet_name=str(config["source_sheet"]),
        header=int(config["pandas_header_index"]),
    )
    source_table.columns = [str(column).strip() for column in source_table.columns]

    source_data_columns = list(source_table.columns)
    source_table["_source_row"] = np.arange(
        int(config["excel_header_row"]) + 1,
        int(config["excel_header_row"]) + 1 + len(source_table),
    )

    non_empty_row_mask = source_table[source_data_columns].notna().any(axis=1)
    source_table = source_table.loc[non_empty_row_mask].copy()
    source_table = source_table.dropna(axis=1, how="all")

    standardized = source_table.rename(columns=SOURCE_TO_STANDARDIZED_COLUMN)
    standardized.insert(0, "source_year", int(config["source_year"]))
    standardized.insert(1, "source_file", str(config["source_file"]))
    standardized.insert(2, "source_sheet", str(config["source_sheet"]))
    standardized.insert(3, "source_row", standardized.pop("_source_row").astype(int))

    duplicate_columns_after_rename = standardized.columns[standardized.columns.duplicated()].tolist()
    if duplicate_columns_after_rename:
        raise ValueError(
            "Source-to-target renaming created duplicate standardized column(s): "
            f"{duplicate_columns_after_rename}"
        )

    for curated_field in LOCKED_CURATED_SCHEMA:
        if curated_field not in standardized.columns:
            standardized[curated_field] = pd.NA

    standardized = standardized[LOCKED_CURATED_SCHEMA].copy()

    # Stabilize sparse later-year integer columns before cross-year concatenation.
    # Older years intentionally contain structural nulls in these columns; explicit
    # nullable integer dtypes keep the concat result deterministic across pandas
    # dtype-inference rules and retain missing values without coercing to float.
    for column_name in CONCAT_STABLE_NULLABLE_INTEGER_FIELDS:
        standardized[column_name] = pd.to_numeric(
            standardized[column_name],
            errors="raise",
        ).astype("Int64")

    return standardized.reset_index(drop=True)

,source_year,source_file,source_sheet,excel_header_row,pandas_header_index
0,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,1,0
1,2021,resultat-ansokningsomgang-2021.xlsx,Tabell 3,1,0
2,2022,resultat-ansokningsomgang-2022.xlsx,Tabell 3,1,0
3,2023,resultat-ansokningsomgang-2023.xlsx,Tabell 3,6,5
4,2024,resultat-ansokningsomgang-2024.xlsx,Tabell 3,6,5
5,2025,resultat-ansokningsomgang-2025.xlsx,Tabell 3,7,6


### 8.2 Standardize every year and concatenate the preliminary applications base

The function is now applied once per configured year. Each yearly result is aligned to the locked 32-field schema before concatenation. Fields that the source workbook does not contain are intentionally created as structural-null placeholders. The normalization and convenience-derived fields remain schema placeholders on the ingestion-stage table; Section 9.2 populates them downstream in `curated_applications` without mutating that ingestion evidence.


In [16]:
standardized_tabell_3_by_year = {
    int(config["source_year"]): read_standardized_tabell_3(config)
    for config in source_import_config.sort_values("source_year").to_dict("records")
}

standardized_applications = pd.concat(
    [standardized_tabell_3_by_year[year] for year in EXPECTED_SOURCE_YEARS],
    ignore_index=True,
)

standardized_row_count_summary = pd.DataFrame(
    [
        {
            "source_year": year,
            "standardized_rows": len(table),
            "standardized_columns": len(table.columns),
        }
        for year, table in standardized_tabell_3_by_year.items()
    ]
).sort_values("source_year").reset_index(drop=True)

display(standardized_row_count_summary)
print(
    "Combined preliminary standardized applications table: "
    f"{standardized_applications.shape[0]:,} rows × {standardized_applications.shape[1]} columns"
)
display(standardized_applications.head())

,source_year,standardized_rows,standardized_columns
0,2020,1482,32
1,2021,1238,32
2,2022,1207,32
3,2023,1258,32
4,2024,1272,32
5,2025,1184,32


Combined preliminary standardized applications table: 7,641 rows × 32 columns


,source_year,source_file,source_sheet,source_row,diarienummer,utbildningsnamn,utbildningsomrade,beslut,beslut_normalized,is_approved,lan,kommun,flera_kommuner,has_multiple_municipalities,antal_kommuner,yh_poang,studieform,is_distance_based,studietakt_procent,examenstyp,utbildningsanordnare,huvudmannatyp,huvudmannatyp_normalized,sokta_utbildningsomgangar,beviljade_utbildningsomgangar,sun5_inriktning,sun5_inriktning_namn,seqf_niva,smalt_yrkesomrade,sokta_platser_per_utbildningsomgang,sokta_platser_totalt,beviljade_platser_totalt
0,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,2,MYH 2020/4419,.NET Developer,Data/IT,Ej beviljad,<NA>,<NA>,Flera kommuner,Flera kommuner,Ja,<NA>,5,425,Bunden,<NA>,100,<NA>,KYH AB,Privat,<NA>,5,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,3,MYH 2020/4482,.NET Developer,Data/IT,Ej beviljad,<NA>,<NA>,Skåne,Malmö,Nej,<NA>,1,430,Bunden,<NA>,100,<NA>,KYH AB Malmö,Privat,<NA>,3,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,4,MYH 2020/5610,.net utvecklare,Data/IT,Ej beviljad,<NA>,<NA>,Västra Götaland,Göteborg,Nej,<NA>,1,400,Bunden,<NA>,100,<NA>,ABF Göteborg Vuxenutbildning AB,Privat,<NA>,3,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,5,MYH 2020/4403,.NET Utvecklare,Data/IT,Beviljad,<NA>,<NA>,Västra Götaland,Göteborg,Nej,<NA>,1,400,Bunden,<NA>,100,<NA>,Plushögskolan AB - Teknikhögskolan,Privat,<NA>,5,3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,6,MYH 2020/5766,.NET-utvecklare,Data/IT,Beviljad,<NA>,<NA>,Stockholm,Stockholm,Nej,<NA>,1,400,Distans,<NA>,100,<NA>,IT-Högskolan Stockholm AB,Privat,<NA>,3,3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


### 8.3 Ingestion integrity checks

These checks are intentionally ingestion-scoped rather than the full final quality-review layer reserved for Sub-project 2.6. They confirm that the standardized import layer obeys the 2.3 schema contract before the later cleaning/enrichment work begins.

In [17]:
expected_row_counts = tabell_3_profile[["source_year", "tabell_3_rows"]].rename(
    columns={"tabell_3_rows": "expected_rows"}
)
observed_row_counts = standardized_applications.groupby("source_year").size().rename("observed_rows").reset_index()
row_count_validation = expected_row_counts.merge(observed_row_counts, on="source_year", how="outer")
row_count_validation["row_count_matches"] = (
    row_count_validation["expected_rows"] == row_count_validation["observed_rows"]
)

combined_schema_order_ok = list(standardized_applications.columns) == LOCKED_CURATED_SCHEMA
per_year_schema_order_ok = all(
    list(table.columns) == LOCKED_CURATED_SCHEMA
    for table in standardized_tabell_3_by_year.values()
)
portable_source_file_ok = standardized_applications["source_file"].map(
    lambda value: str(value) == Path(str(value)).name and "/" not in str(value) and "\\" not in str(value)
).all()
source_sheet_ok = standardized_applications["source_sheet"].eq("Tabell 3").all()
source_year_scope_ok = sorted(standardized_applications["source_year"].unique().tolist()) == EXPECTED_SOURCE_YEARS
source_row_present_ok = standardized_applications["source_row"].notna().all()
missing_diarienummer_count = int(standardized_applications["diarienummer"].isna().sum())
duplicate_application_key_count = int(
    standardized_applications.duplicated(["source_year", "diarienummer"]).sum()
)

later_year_only_fields = [
    "sun5_inriktning",
    "sun5_inriktning_namn",
    "seqf_niva",
    "smalt_yrkesomrade",
    "sokta_platser_per_utbildningsomgang",
    "sokta_platser_totalt",
    "beviljade_platser_totalt",
]

examenstyp_structural_null_ok = standardized_applications.loc[
    standardized_applications["source_year"] == 2020,
    "examenstyp",
].isna().all()

later_year_structural_null_ok = standardized_applications.loc[
    standardized_applications["source_year"].isin([2020, 2021, 2022]),
    later_year_only_fields,
].isna().all().all()

ingestion_placeholder_fields_unpopulated_ok = standardized_applications[
    INGESTION_STAGE_PLACEHOLDER_FIELDS
].isna().all().all()

ingestion_validation_summary = pd.DataFrame(
    [
        {"check": "Locked 32-column schema order on combined table", "passed": combined_schema_order_ok},
        {"check": "Locked schema order on every yearly table", "passed": per_year_schema_order_ok},
        {"check": "Configured source years match 2020–2025", "passed": source_year_scope_ok},
        {"check": "Portable source_file values use filenames only", "passed": portable_source_file_ok},
        {"check": "source_sheet is Tabell 3 for every row", "passed": source_sheet_ok},
        {"check": "source_row populated for every standardized row", "passed": source_row_present_ok},
        {"check": "No missing Diarienummer", "passed": missing_diarienummer_count == 0},
        {"check": "No duplicate (source_year, diarienummer) keys", "passed": duplicate_application_key_count == 0},
        {"check": "2020 examenstyp structural nulls preserved", "passed": examenstyp_structural_null_ok},
        {"check": "2020–2022 later-year-only structural nulls preserved", "passed": later_year_structural_null_ok},
        {"check": "Ingestion-stage normalization and convenience fields remain unpopulated", "passed": ingestion_placeholder_fields_unpopulated_ok},
    ]
)

display(row_count_validation)
display(ingestion_validation_summary)

assert row_count_validation["row_count_matches"].all(), (
    "Each standardized yearly table must preserve the profiled Tabell 3 row count."
)
assert combined_schema_order_ok and per_year_schema_order_ok, (
    "Standardized ingestion must enforce the locked 32-field schema order."
)
assert source_year_scope_ok, "The standardized applications table must cover exactly the expected 2020–2025 years."
assert portable_source_file_ok, "source_file should contain portable filenames only, not absolute paths."
assert source_sheet_ok, "The standardized ingestion table should only contain Tabell 3 rows."
assert source_row_present_ok, "source_row must be populated for every standardized row."
assert missing_diarienummer_count == 0, "Unexpected missing Diarienummer after standardized ingestion."
assert duplicate_application_key_count == 0, "Unexpected duplicate application keys after standardized ingestion."
assert examenstyp_structural_null_ok, "2020 should retain structural nulls for examenstyp."
assert later_year_structural_null_ok, "2020–2022 should retain structural nulls for later-year-only fields."
assert ingestion_placeholder_fields_unpopulated_ok, (
    "Normalization and selected convenience fields must remain unpopulated in standardized_applications."
)

,source_year,expected_rows,observed_rows,row_count_matches
0,2020,1482,1482,True
1,2021,1238,1238,True
2,2022,1207,1207,True
3,2023,1258,1258,True
4,2024,1272,1272,True
5,2025,1184,1184,True


,check,passed
0,Locked 32-column schema order on combined table,True
1,Locked schema order on every yearly table,True
2,Configured source years match 2020–2025,True
3,Portable source_file values use filenames only,True
4,source_sheet is Tabell 3 for every row,True
5,source_row populated for every standardized row,True
6,No missing Diarienummer,True
7,"No duplicate (source_year, diarienummer) keys",True
8,2020 examenstyp structural nulls preserved,True
9,2020–2022 later-year-only structural nulls pre...,True


## 9. Cleaning, normalization, and enrichment *(Sub-project 2.5)*

The standardized ingestion table is intentionally close to the source. This section performs the next layer of the workflow:
- convert imported representations into more analysis-ready datatypes,
- remove only whitespace artifacts that do not carry source meaning,
- populate the normalization fields and derived boolean fields already specified in Sub-project 2.3.

The separation matters. The ingestion layer proves that all six raw Excel workbooks can be read and aligned without interpretation. The cleaning/enrichment layer then makes that aligned table more consistent for longitudinal comparison, later validation, and later SQL/API use.

### 9.1 Datatype conversion and safe text cleanup

The combined `standardized_applications` table already has stable field names because the Sub-project 2.4 ingestion reader pre-stabilizes its four structurally sparse later-year integer fields as nullable `Int64` before cross-year concatenation. This 2.5 cleaning layer keeps that inherited dtype contract explicit and improves source-text comparison safety:
- `seqf_niva`, `sokta_platser_per_utbildningsomgang`, `sokta_platser_totalt`, and `beviljade_platser_totalt` must retain nullable `Int64` dtypes so structural nulls remain valid,
- several text fields contain harmless but comparison-breaking whitespace noise, such as trailing spaces in names.

The cleanup here is deliberately conservative:
- trim leading/trailing whitespace,
- collapse repeated whitespace to a single space,
- preserve missing values as missing,
- avoid case changes, accent removal, or semantic rewriting.

This improves equality checks and downstream grouping without changing the substantive source text. The text helper also moves cleaned text fields into pandas' nullable `string` dtype, while the numeric step revalidates the sparse `Int64` contract rather than relying on version-sensitive concat inference.


In [18]:
TEXT_COLUMNS_TO_CLEAN = [
    "source_file",
    "source_sheet",
    "diarienummer",
    "utbildningsnamn",
    "utbildningsomrade",
    "beslut",
    "lan",
    "kommun",
    "flera_kommuner",
    "studieform",
    "examenstyp",
    "utbildningsanordnare",
    "huvudmannatyp",
    "sun5_inriktning",
    "sun5_inriktning_namn",
    "smalt_yrkesomrade",
]

NULLABLE_INTEGER_COLUMNS = [
    "seqf_niva",
    "sokta_platser_per_utbildningsomgang",
    "sokta_platser_totalt",
    "beviljade_platser_totalt",
]


def clean_text_values(series: pd.Series) -> pd.Series:
    """Return conservatively cleaned nullable text without changing substantive wording."""
    cleaned = series.astype("string")
    cleaned = cleaned.str.replace(r"\s+", " ", regex=True).str.strip()
    return cleaned.mask(cleaned.eq(""), pd.NA)


def convert_to_nullable_integer(series: pd.Series, column_name: str) -> pd.Series:
    """Convert integer-like imported values to nullable Int64 while preserving structural nulls."""
    numeric = pd.to_numeric(series, errors="raise")
    non_null_values = numeric.dropna()
    non_integer_values = non_null_values[~np.isclose(non_null_values % 1, 0)]
    if not non_integer_values.empty:
        raise ValueError(
            f"{column_name} contains non-integer numeric value(s): "
            f"{sorted(non_integer_values.unique().tolist())[:5]}"
        )
    return numeric.astype("Int64")


def apply_cleaning_layer(standardized_input: pd.DataFrame) -> pd.DataFrame:
    """Create the cleaned 2.5 working table without mutating standardized ingestion output."""
    cleaned = standardized_input.copy(deep=True)

    for column_name in TEXT_COLUMNS_TO_CLEAN:
        cleaned[column_name] = clean_text_values(cleaned[column_name])

    for column_name in NULLABLE_INTEGER_COLUMNS:
        cleaned[column_name] = convert_to_nullable_integer(cleaned[column_name], column_name)

    return cleaned


cleaned_applications = apply_cleaning_layer(standardized_applications)

datatype_conversion_summary = pd.DataFrame(
    [
        {
            "column": column_name,
            "dtype_before_cleaning": str(standardized_applications[column_name].dtype),
            "dtype_after_cleaning": str(cleaned_applications[column_name].dtype),
            "non_null_values_after_cleaning": int(cleaned_applications[column_name].notna().sum()),
        }
        for column_name in NULLABLE_INTEGER_COLUMNS
    ]
)

text_cleanup_summary = pd.DataFrame(
    [
        {
            "column": column_name,
            "changed_values": int(
                (
                    standardized_applications[column_name].astype("string").fillna("<NA>")
                    != cleaned_applications[column_name].astype("string").fillna("<NA>")
                ).sum()
            ),
            "dtype_after_cleaning": str(cleaned_applications[column_name].dtype),
        }
        for column_name in TEXT_COLUMNS_TO_CLEAN
    ]
).sort_values(["changed_values", "column"], ascending=[False, True]).reset_index(drop=True)

print(
    "Cleaned applications working table: "
    f"{cleaned_applications.shape[0]:,} rows × {cleaned_applications.shape[1]} columns"
)
display(datatype_conversion_summary)
display(text_cleanup_summary.loc[text_cleanup_summary["changed_values"] > 0])

Cleaned applications working table: 7,641 rows × 32 columns


,column,dtype_before_cleaning,dtype_after_cleaning,non_null_values_after_cleaning
0,seqf_niva,Int64,Int64,3661
1,sokta_platser_per_utbildningsomgang,Int64,Int64,3714
2,sokta_platser_totalt,Int64,Int64,3714
3,beviljade_platser_totalt,Int64,Int64,3714


,column,changed_values,dtype_after_cleaning
0,utbildningsnamn,565,string
1,utbildningsanordnare,109,string
2,sun5_inriktning_namn,6,string


### 9.2 Normalization and useful derived fields

The 2.3 design intentionally retained both source-facing categories and normalized analytical companions. That pattern is implemented here:
- `beslut` remains the original source decision wording, while `beslut_normalized` becomes a stable longitudinal category,
- `huvudmannatyp` remains the source wording, while `huvudmannatyp_normalized` harmonizes `Landsting` and `Region`,
- three boolean convenience fields are populated so common questions become direct filters rather than repeated notebook logic.

The mapping helper validates coverage before assigning values. If a future workbook introduces an unexpected non-null category, the notebook should fail clearly instead of silently creating partial mappings.

In [19]:
STUDIEFORM_TO_IS_DISTANCE_BASED = {
    "Distans": True,
    "Bunden": False,
}

FLERA_KOMMUNER_TO_BOOLEAN = {
    "Ja": True,
    "Nej": False,
}

BESLUT_NORMALIZED_TO_IS_APPROVED = {
    "approved": True,
    "rejected": False,
    "withdrawn": False,
}


def map_values_with_coverage(
    series: pd.Series,
    mapping: dict[str, object],
    field_name: str,
) -> pd.Series:
    """Map non-null source categories only when every observed value is explicitly covered."""
    observed_values = {str(value) for value in series.dropna().unique()}
    unmapped_values = sorted(observed_values - set(mapping))
    if unmapped_values:
        raise ValueError(f"{field_name} contains unmapped value(s): {unmapped_values}")
    return series.map(mapping)


def add_normalization_and_enrichment(cleaned_input: pd.DataFrame) -> pd.DataFrame:
    """Populate the normalized categories and 2.5 boolean convenience fields."""
    curated = cleaned_input.copy(deep=True)

    curated["beslut_normalized"] = map_values_with_coverage(
        curated["beslut"],
        BESLUT_NORMALIZATION,
        "beslut",
    ).astype("string")

    curated["huvudmannatyp_normalized"] = map_values_with_coverage(
        curated["huvudmannatyp"],
        HUVUDMANNATYP_NORMALIZATION,
        "huvudmannatyp",
    ).astype("string")

    curated["is_approved"] = map_values_with_coverage(
        curated["beslut_normalized"],
        BESLUT_NORMALIZED_TO_IS_APPROVED,
        "beslut_normalized",
    ).astype("boolean")

    curated["is_distance_based"] = map_values_with_coverage(
        curated["studieform"],
        STUDIEFORM_TO_IS_DISTANCE_BASED,
        "studieform",
    ).astype("boolean")

    curated["has_multiple_municipalities"] = map_values_with_coverage(
        curated["flera_kommuner"],
        FLERA_KOMMUNER_TO_BOOLEAN,
        "flera_kommuner",
    ).astype("boolean")

    return curated


curated_applications = add_normalization_and_enrichment(cleaned_applications)

beslut_normalization_result = (
    curated_applications.groupby(["beslut", "beslut_normalized"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["beslut_normalized", "beslut"])
    .reset_index(drop=True)
)

huvudmannatyp_normalization_result = (
    curated_applications.groupby(["huvudmannatyp", "huvudmannatyp_normalized"], dropna=False)
    .size()
    .rename("rows")
    .reset_index()
    .sort_values(["huvudmannatyp_normalized", "huvudmannatyp"])
    .reset_index(drop=True)
)

derived_boolean_summary = pd.DataFrame(
    [
        {
            "field": column_name,
            "true_rows": int(curated_applications[column_name].eq(True).sum()),
            "false_rows": int(curated_applications[column_name].eq(False).sum()),
            "missing_rows": int(curated_applications[column_name].isna().sum()),
            "dtype": str(curated_applications[column_name].dtype),
        }
        for column_name in [
            "is_approved",
            "is_distance_based",
            "has_multiple_municipalities",
        ]
    ]
)

print(
    "Curated 2.5 applications table: "
    f"{curated_applications.shape[0]:,} rows × {curated_applications.shape[1]} columns"
)
display(beslut_normalization_result)
display(huvudmannatyp_normalization_result)
display(derived_boolean_summary)

Curated 2.5 applications table: 7,641 rows × 32 columns


,beslut,beslut_normalized,rows
0,Beviljad,approved,2613
1,Avslag,rejected,3217
2,Ej beviljad,rejected,1810
3,Återkallad,withdrawn,1


,huvudmannatyp,huvudmannatyp_normalized,rows
0,Kommun,Kommun,1261
1,Privat,Privat,6284
2,Landsting,Region,23
3,Region,Region,60
4,Statlig,Statlig,13


,field,true_rows,false_rows,missing_rows,dtype
0,is_approved,2613,5028,0,boolean
1,is_distance_based,3562,4079,0,boolean
2,has_multiple_municipalities,1433,6208,0,boolean


### 9.3 Rerun safety and transformation integrity checks

The 2.5 layer must be safe to rerun from the same standardized input. Instead of mutating the ingestion-stage table, the notebook uses copy-based transformation functions and verifies that:
- repeated execution produces the same curated table,
- the 32-field schema and row count stay unchanged,
- the original `standardized_applications` table remains unpopulated in the fields that 2.5 is responsible for filling,
- conservative text cleanup actually removes outer/repeated whitespace,
- nullable integer and nullable boolean dtypes are present where intended,
- the 2.5 normalized and derived fields are populated completely for the observed source categories.

These checks are transformation-scoped. The broader dataset validation, missingness review, export work, and final quality report remain intentionally reserved for Sub-project 2.6.

In [20]:
curated_applications_rerun_check = add_normalization_and_enrichment(
    apply_cleaning_layer(standardized_applications)
)

transformation_rerun_safe_ok = curated_applications.equals(curated_applications_rerun_check)
row_count_preserved_after_cleaning_ok = len(curated_applications) == len(standardized_applications)
schema_order_preserved_after_cleaning_ok = list(curated_applications.columns) == LOCKED_CURATED_SCHEMA
standardized_input_preserved_ok = standardized_applications[
    INGESTION_STAGE_PLACEHOLDER_FIELDS
].isna().all().all()

standardized_sparse_integer_dtypes_ok = all(
    str(standardized_applications[column_name].dtype) == "Int64"
    for column_name in CONCAT_STABLE_NULLABLE_INTEGER_FIELDS
)

cleaned_text_whitespace_ok = all(
    not curated_applications[column_name]
    .dropna()
    .astype("string")
    .str.contains(r"^\s|\s$|\s{2,}", regex=True)
    .any()
    for column_name in TEXT_COLUMNS_TO_CLEAN
)

nullable_integer_dtypes_ok = all(
    str(curated_applications[column_name].dtype) == "Int64"
    for column_name in NULLABLE_INTEGER_COLUMNS
)

nullable_boolean_dtypes_ok = all(
    str(curated_applications[column_name].dtype) == "boolean"
    for column_name in [
        "is_approved",
        "is_distance_based",
        "has_multiple_municipalities",
    ]
)

normalized_and_derived_fields_populated_ok = curated_applications[
    [
        "beslut_normalized",
        "huvudmannatyp_normalized",
        "is_approved",
        "is_distance_based",
        "has_multiple_municipalities",
    ]
].notna().all().all()

normalized_value_domains_ok = (
    set(curated_applications["beslut_normalized"].dropna().unique())
    <= set(BESLUT_NORMALIZATION.values())
    and set(curated_applications["huvudmannatyp_normalized"].dropna().unique())
    <= set(HUVUDMANNATYP_NORMALIZATION.values())
)

transformation_validation_summary = pd.DataFrame(
    [
        {"check": "Repeated 2.5 transformation gives the same curated table", "passed": transformation_rerun_safe_ok},
        {"check": "2.5 transformation preserves the 7,641-row input count", "passed": row_count_preserved_after_cleaning_ok},
        {"check": "2.5 transformation preserves locked 32-field schema order", "passed": schema_order_preserved_after_cleaning_ok},
        {"check": "Standardized ingestion table remains unmodified in 2.5-derived fields", "passed": standardized_input_preserved_ok},
        {"check": "Standardized sparse later-year integer fields are concat-stable nullable Int64", "passed": standardized_sparse_integer_dtypes_ok},
        {"check": "Safe text cleanup removes outer and repeated whitespace", "passed": cleaned_text_whitespace_ok},
        {"check": "Selected later-year numeric fields use nullable Int64", "passed": nullable_integer_dtypes_ok},
        {"check": "Derived convenience flags use nullable boolean dtype", "passed": nullable_boolean_dtypes_ok},
        {"check": "Normalized and derived 2.5 fields are populated for observed categories", "passed": normalized_and_derived_fields_populated_ok},
        {"check": "Normalized value domains remain within the locked 2.3 mappings", "passed": normalized_value_domains_ok},
    ]
)

display(transformation_validation_summary)
display(curated_applications.head())

assert transformation_rerun_safe_ok, "Repeating the 2.5 transformation should reproduce the same curated table."
assert row_count_preserved_after_cleaning_ok, "Cleaning/enrichment should not add or remove application rows."
assert schema_order_preserved_after_cleaning_ok, "Cleaning/enrichment must preserve the locked 32-field schema order."
assert standardized_input_preserved_ok, "The 2.5 layer should not mutate standardized_applications in place."
assert standardized_sparse_integer_dtypes_ok, "Sparse later-year integer fields should be nullable Int64 before and after concatenation."
assert cleaned_text_whitespace_ok, "Text cleanup should remove outer and repeated whitespace from configured text columns."
assert nullable_integer_dtypes_ok, "Selected later-year numeric fields should use nullable Int64 after cleaning."
assert nullable_boolean_dtypes_ok, "Derived convenience flags should use pandas nullable boolean dtype."
assert normalized_and_derived_fields_populated_ok, "Observed 2.5 normalization and boolean outputs should be fully populated."
assert normalized_value_domains_ok, "Normalized values must stay inside the locked 2.3 mapping domains."

,check,passed
0,Repeated 2.5 transformation gives the same cur...,True
1,"2.5 transformation preserves the 7,641-row inp...",True
2,2.5 transformation preserves locked 32-field s...,True
3,Standardized ingestion table remains unmodifie...,True
4,Standardized sparse later-year integer fields ...,True
5,Safe text cleanup removes outer and repeated w...,True
6,Selected later-year numeric fields use nullabl...,True
7,Derived convenience flags use nullable boolean...,True
8,Normalized and derived 2.5 fields are populate...,True
9,Normalized value domains remain within the loc...,True


,source_year,source_file,source_sheet,source_row,diarienummer,utbildningsnamn,utbildningsomrade,beslut,beslut_normalized,is_approved,lan,kommun,flera_kommuner,has_multiple_municipalities,antal_kommuner,yh_poang,studieform,is_distance_based,studietakt_procent,examenstyp,utbildningsanordnare,huvudmannatyp,huvudmannatyp_normalized,sokta_utbildningsomgangar,beviljade_utbildningsomgangar,sun5_inriktning,sun5_inriktning_namn,seqf_niva,smalt_yrkesomrade,sokta_platser_per_utbildningsomgang,sokta_platser_totalt,beviljade_platser_totalt
0,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,2,MYH 2020/4419,.NET Developer,Data/IT,Ej beviljad,rejected,False,Flera kommuner,Flera kommuner,Ja,True,5,425,Bunden,False,100,<NA>,KYH AB,Privat,Privat,5,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
1,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,3,MYH 2020/4482,.NET Developer,Data/IT,Ej beviljad,rejected,False,Skåne,Malmö,Nej,False,1,430,Bunden,False,100,<NA>,KYH AB Malmö,Privat,Privat,3,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
2,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,4,MYH 2020/5610,.net utvecklare,Data/IT,Ej beviljad,rejected,False,Västra Götaland,Göteborg,Nej,False,1,400,Bunden,False,100,<NA>,ABF Göteborg Vuxenutbildning AB,Privat,Privat,3,0,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
3,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,5,MYH 2020/4403,.NET Utvecklare,Data/IT,Beviljad,approved,True,Västra Götaland,Göteborg,Nej,False,1,400,Bunden,False,100,<NA>,Plushögskolan AB - Teknikhögskolan,Privat,Privat,5,3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>
4,2020,resultat-ansokningsomgang-2020.xlsx,Tabell 3,6,MYH 2020/5766,.NET-utvecklare,Data/IT,Beviljad,approved,True,Stockholm,Stockholm,Nej,False,1,400,Distans,True,100,<NA>,IT-Högskolan Stockholm AB,Privat,Privat,3,3,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>


## 10. Validation and quality checks *(Sub-project 2.6)*

This section will later implement stronger-than-minimal confidence checks, such as:
- row counts by source year,
- missing identifier checks,
- duplicate `diarienummer` checks within each year,
- normalized-value mapping coverage,
- missingness review,
- datatype checks,
- sanity summaries by year and category.


## 11. Export of the curated dataset *(Sub-project 2.6)*

The final curated dataset will be written into:

```text
part_2/data/processed/
```

CSV is the expected baseline export.  
Whether a Parquet export adds enough value to keep will be decided later, not prematurely.


## 12. SQL/API handoff note and final reflection *(Sub-project 2.7)*

This final section will explain:
- what table was exported,
- its grain and intended downstream use,
- how it supports later SQL loading,
- how it can support simple read-oriented FastAPI endpoints,
- what limitations or future extensions remain.


## Sub-project 2.5 checkpoint

The cleaning, normalization, and thoughtful enrichment phase is complete when:
- the inherited 2.4 ingestion reader keeps the four structurally sparse later-year integer fields concat-stable as nullable `Int64` before cross-year concatenation,
- `standardized_applications` remains the untouched ingestion-stage base from Sub-project 2.4,
- the ingestion-stage normalization and convenience placeholders remain unpopulated inside `standardized_applications`,
- `apply_cleaning_layer(...)` creates `cleaned_applications` without mutating the ingestion table,
- configured text fields use conservative whitespace cleanup only,
- `seqf_niva` and the three later-year seat-total fields keep nullable `Int64`, preserving structural nulls,
- `add_normalization_and_enrichment(...)` creates `curated_applications`,
- `beslut_normalized` and `huvudmannatyp_normalized` follow the locked 2.3 mapping rules,
- `is_approved`, `is_distance_based`, and `has_multiple_municipalities` are populated as nullable boolean fields,
- transformation-scoped assertions confirm rerun safety, schema-order preservation, row-count preservation, input-table preservation, dtype intent, text-cleanup intent, and allowed normalization domains,
- the next implementation boundary is Sub-project **2.6 — Validation, quality checks, and export**.
